# Final CatBoost Model Pipeline

This notebook provides a compact, reproducible inference pipeline for the final model selected in Issue #45.

The selected model is a `CatBoostRegressor` using the top 75 ranked predictors. Model selection, cross-validation, feature ranking, feature-selection experiments, and tuning were completed separately and are not repeated here.

This pipeline:

1. verifies the frozen modeling source and split;
2. loads the selected 75-feature list;
3. prepares the categorical fields expected by CatBoost;
4. loads the saved fitted CatBoost model;
5. reproduces predictions on the frozen validation partition;
6. calculates MAE and basic prediction checks;
7. saves a compact pipeline receipt.

The protected test partition remains unused.

In [1]:
from pathlib import Path
from importlib.metadata import version
import hashlib
import json
import platform

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error

## 1. Frozen Inputs and Artifact Paths

The pipeline uses the same frozen source and partition manifest used throughout October modeling.

The fitted model and selected feature list are loaded from the completed CatBoost experiments rather than reconstructed or retuned.

In [2]:
SOURCE_SHA256 = "74037cb248a1064e4d578692a4f4e5d8492ed1b2033daf643496e1b68b14ae03"
MANIFEST_SHA256 = "de10b0e17ad56174a7b126352a0a41b3b3d477bbddd65f37b87a6235f582a64b"

TRAIN_COUNT = 131_822
VALIDATION_COUNT = 28_248

CAT_COLS = [f"cat{i}" for i in range(1, 117)]

repo_root = Path.cwd()

while not (repo_root / "data" / "allstate_claims_data.csv").exists():
    if repo_root.parent == repo_root:
        raise FileNotFoundError("Could not locate repository root.")
    repo_root = repo_root.parent

source_path = (
    repo_root
    / "data"
    / "allstate_claims_data.csv"
)

manifest_path = (
    repo_root
    / "notebooks"
    / "final-deliverables"
    / "October"
    / "Gate 1"
    / "partition_manifest.csv"
)

model_artifact_dir = (
    repo_root
    / "notebooks"
    / "liam-stapley"
    / "model_experiment_outputs"
)

model_path = (
    model_artifact_dir
    / "final_top75_catboost_model.cbm"
)

feature_path = (
    model_artifact_dir
    / "final_catboost_features.csv"
)

saved_predictions_path = (
    model_artifact_dir
    / "final_catboost_validation_predictions.csv"
)

output_dir = (
    repo_root
    / "notebooks"
    / "final-deliverables"
    / "October"
    / "Gate 3"
    / "final_model_pipeline_results"
)

output_dir.mkdir(parents=True, exist_ok=True)

print("Repository:", repo_root)
print("Model:", model_path)

Repository: c:\Users\stapl\BTT\Fall AI Studio\Allstate 1A\Allstate-1A-predicting-auto-claims-severity
Model: c:\Users\stapl\BTT\Fall AI Studio\Allstate 1A\Allstate-1A-predicting-auto-claims-severity\notebooks\liam-stapley\model_experiment_outputs\final_top75_catboost_model.cbm


## 2. Verify Frozen Data

The source file and partition manifest are verified before any prediction is made.

This prevents the final pipeline from silently running against a different dataset or split than the one used during model selection.

In [3]:
def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


assert sha256_file(source_path) == SOURCE_SHA256
assert sha256_file(manifest_path) == MANIFEST_SHA256

assert model_path.exists()
assert feature_path.exists()

print("Frozen inputs and saved model artifacts verified.")

Frozen inputs and saved model artifacts verified.


## 3. Load the Frozen Validation Partition

The pipeline reads the frozen source and partition manifest, aligns them by claim ID, and exposes only the validation partition for this reproducibility check.

The protected test partition is not used.

In [4]:
data = pd.read_csv(source_path)
manifest = pd.read_csv(manifest_path)

df = data.merge(
    manifest,
    on="id",
    how="inner",
    validate="one_to_one",
)

assert len(df[df["partition"] == "train"]) == TRAIN_COUNT
assert len(df[df["partition"] == "validation"]) == VALIDATION_COUNT

validation = (
    df[df["partition"] == "validation"]
    .copy()
    .reset_index(drop=True)
)

validation_ids = validation["id"].copy()
y_validation = validation["loss"].copy()

print("Validation claims:", len(validation))

Validation claims: 28248


## 4. Prepare the Selected Features

The final model uses the 75 predictors selected during the feature-ranking experiments.

No scaling, one-hot encoding, or target transformation is applied. CatBoost handles the selected categorical predictors natively.

Categorical values are converted to strings to reproduce the representation used during training.

In [5]:
selected_features = (
    pd.read_csv(feature_path)["feature"]
    .tolist()
)

assert len(selected_features) == 75
assert len(set(selected_features)) == 75

missing_features = [
    feature
    for feature in selected_features
    if feature not in validation.columns
]

assert not missing_features, (
    f"Missing selected features: {missing_features}"
)

selected_cat_cols = [
    feature
    for feature in selected_features
    if feature in CAT_COLS
]


def prepare_catboost_features(frame):
    X = frame[selected_features].copy()

    for column in selected_cat_cols:
        X[column] = X[column].astype(str)

    return X


X_validation = prepare_catboost_features(validation)

print("Features:", X_validation.shape[1])
print("Categorical features:", len(selected_cat_cols))

Features: 75
Categorical features: 61


## 5. Load the Final Model

The fitted `.cbm` artifact is loaded directly.

The model is not retrained in this notebook. This keeps the final pipeline separate from the exploratory model-selection process and avoids repeating the expensive five-fold CatBoost experiments.

In [6]:
model = CatBoostRegressor()
model.load_model(str(model_path))

print("Saved CatBoost model loaded successfully.")

Saved CatBoost model loaded successfully.


## 6. Reproduce Validation Predictions

The saved model is evaluated on the same 28,248 validation claims used during model comparison.

MAE remains in original `loss` units.

In [7]:
predictions = model.predict(X_validation)

validation_mae = mean_absolute_error(
    y_validation,
    predictions,
)

assert len(predictions) == VALIDATION_COUNT
assert np.isfinite(predictions).all()

print(f"Validation MAE: {validation_mae:.6f}")
print("Finite predictions:", np.isfinite(predictions).all())
print("Negative predictions:", int((predictions < 0).sum()))

Validation MAE: 1136.341878
Finite predictions: True
Negative predictions: 0


## 7. Reproducibility Check

If saved validation predictions are available from the model-selection notebook, this section confirms that loading the final `.cbm` model reproduces them.

This provides a direct check that the saved model artifact is the same fitted model used for the reported final results.

In [8]:
if saved_predictions_path.exists():
    saved_predictions = pd.read_csv(
        saved_predictions_path
    )

    saved_prediction_column = (
        "prediction"
        if "prediction" in saved_predictions.columns
        else "reduced_catboost_prediction"
    )

    saved_lookup = (
        saved_predictions[
            ["id", saved_prediction_column]
        ]
        .rename(
            columns={
                saved_prediction_column:
                "saved_prediction"
            }
        )
    )

    reproduced = pd.DataFrame(
        {
            "id": validation_ids,
            "prediction": predictions,
        }
    ).merge(
        saved_lookup,
        on="id",
        how="inner",
        validate="one_to_one",
    )

    max_difference = np.max(
        np.abs(
            reproduced["prediction"]
            - reproduced["saved_prediction"]
        )
    )

    print(
        "Maximum difference from saved predictions:",
        max_difference,
    )

else:
    print(
        "Saved validation prediction artifact not found; "
        "skipping prediction cross-check."
    )

Maximum difference from saved predictions: 3.637978807091713e-12


## 8. Save Pipeline Evidence

The final predictions and a compact run receipt are saved so the pipeline can be independently checked without repeating model training.

In [9]:
pipeline_predictions = pd.DataFrame(
    {
        "id": validation_ids,
        "loss": y_validation,
        "prediction": predictions,
        "absolute_error": np.abs(
            y_validation - predictions
        ),
    }
)

prediction_path = (
    output_dir
    / "final_pipeline_validation_predictions.csv"
)

pipeline_predictions.to_csv(
    prediction_path,
    index=False,
)

receipt = {
    "modeling_data_version": "OCT-MD-001",
    "split_version": "OCT-SPLIT-001",
    "source_sha256": SOURCE_SHA256,
    "manifest_sha256": MANIFEST_SHA256,
    "model_path": str(
        model_path.relative_to(repo_root)
    ),
    "model_sha256": sha256_file(model_path),
    "feature_path": str(
        feature_path.relative_to(repo_root)
    ),
    "feature_count": len(selected_features),
    "categorical_feature_count": len(
        selected_cat_cols
    ),
    "validation_claims": len(validation),
    "validation_mae": float(validation_mae),
    "metric": "MAE",
    "metric_units": "original loss units",
    "negative_predictions": int(
        (predictions < 0).sum()
    ),
    "finite_predictions": bool(
        np.isfinite(predictions).all()
    ),
    "test_partition_used": False,
    "library_versions": {
        "python": platform.python_version(),
        "numpy": version("numpy"),
        "pandas": version("pandas"),
        "catboost": version("catboost"),
        "scikit-learn": version(
            "scikit-learn"
        ),
    },
}

receipt_path = (
    output_dir
    / "final_model_pipeline_receipt.json"
)

receipt_path.write_text(
    json.dumps(receipt, indent=2) + "\n",
    encoding="utf-8",
)

print("Saved:")
print(prediction_path)
print(receipt_path)

Saved:
c:\Users\stapl\BTT\Fall AI Studio\Allstate 1A\Allstate-1A-predicting-auto-claims-severity\notebooks\final-deliverables\October\Gate 3\final_model_pipeline_results\final_pipeline_validation_predictions.csv
c:\Users\stapl\BTT\Fall AI Studio\Allstate 1A\Allstate-1A-predicting-auto-claims-severity\notebooks\final-deliverables\October\Gate 3\final_model_pipeline_results\final_model_pipeline_receipt.json


## Final Pipeline

The final Issue #45 model can therefore be reproduced without repeating model selection or cross-validation:

`Frozen source + frozen split`
→ `75 selected predictors`
→ `categorical string preparation`
→ `saved CatBoostRegressor`
→ `prediction in original loss units`

The pipeline intentionally contains no unnecessary scaling, feature engineering, or target transformation. Those operations were not part of the selected CatBoost model.